In [ ]:
# 1. 必要なライブラリのインストール
!pip install face_recognition

import base64
import cv2
import face_recognition
import numpy as np
from IPython.display import display, Javascript
from google.colab.output import eval_js
from google.colab.patches import cv2_imshow

# 2. Google Colab用 Webカメラ撮影関数 (JavaScript連携)
def take_photo(quality=0.8):
    """
    Google Colabのブラウザ環境からWebカメラを起動し、
    撮影した画像をOpenCV形式(BGR画像)のNumPy配列として返します。
    """
    js = Javascript('''
        async function takePhoto(quality) {
            const div = document.createElement('div');
            div.style.fontFamily = 'Arial, sans-serif';
            div.style.marginBottom = '12px';

            const message = document.createElement('p');
            message.textContent = 'カメラの前に顔を合わせ、「撮影 (Capture)」ボタンを押してください。';
            message.style.fontWeight = 'bold';
            message.style.color = '#333';

            const capture = document.createElement('button');
            capture.textContent = '撮影 (Capture)';
            capture.style.display = 'block';
            capture.style.margin = '10px 0';
            capture.style.padding = '10px 20px';
            capture.style.fontSize = '16px';
            capture.style.backgroundColor = '#1a73e8';
            capture.style.color = 'white';
            capture.style.border = 'none';
            capture.style.borderRadius = '6px';
            capture.style.cursor = 'pointer';
            capture.style.boxShadow = '0 2px 4px rgba(0,0,0,0.2)';

            const video = document.createElement('video');
            video.style.display = 'block';
            video.style.borderRadius = '8px';
            video.style.boxShadow = '0 4px 10px rgba(0,0,0,0.15)';
            video.style.transform = 'scaleX(-1)'; // 鏡像表示（自撮り用）

            const stream = await navigator.mediaDevices.getUserMedia({video: true});

            document.body.appendChild(div);
            div.appendChild(message);
            div.appendChild(video);
            div.appendChild(capture);
            video.srcObject = stream;
            await video.play();

            // Colabの出力高さを調整
            google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);

            // 「撮影」ボタンが押されるまで待機
            await new Promise((resolve) => capture.onclick = resolve);

            const canvas = document.createElement('canvas');
            canvas.width = video.videoWidth;
            canvas.height = video.videoHeight;
            const ctx = canvas.getContext('2d');
            
            // 鏡像を維持して画像キャプチャ
            ctx.translate(canvas.width, 0);
            ctx.scale(-1, 1);
            ctx.drawImage(video, 0, 0);

            // ストリーム停止とUI要素の削除
            stream.getVideoTracks()[0].stop();
            div.remove();

            return canvas.toDataURL('image/jpeg', quality);
        }
    ''')
    display(js)
    data = eval_js(f'takePhoto({quality})')

    # Base64文字列をデコードしてOpenCV(BGR)画像に変換
    header, encoded = data.split(',', 1)
    binary = base64.b64decode(encoded)
    image_np = np.frombuffer(binary, dtype=np.uint8)
    image_bgr = cv2.imdecode(image_np, cv2.IMREAD_COLOR)
    
    return image_bgr

# 3. 目の「開き具合（EAR: Eye Aspect Ratio）」を計算する関数
def calculate_ear(eye_landmarks):
    """
    目の6つの特徴点座標からEAR(Eye Aspect Ratio)を計算します。
    EAR = (|p2 - p6| + |p3 - p5|) / (2 * |p1 - p4|)
    """
    # まぶたの上下の距離（垂直方向）を計算
    p2_p6 = np.linalg.norm(np.array(eye_landmarks[1]) - np.array(eye_landmarks[5]))
    p3_p5 = np.linalg.norm(np.array(eye_landmarks[2]) - np.array(eye_landmarks[4]))
    # 目頭と目尻の距離（水平方向）を計算
    p1_p4 = np.linalg.norm(np.array(eye_landmarks[0]) - np.array(eye_landmarks[3]))

    # ゼロ除算対策
    if p1_p4 == 0:
        return 0.0

    # 目の開き具合の比率（EAR）を算出
    ear = (p2_p6 + p3_p5) / (2.0 * p1_p4)
    return ear

# 4. メイン処理：Webカメラから画像を取得して判定
print("Webカメラを起動しています...")
try:
    image_bgr = take_photo()
except Exception as e:
    print(f"エラー: Webカメラからの画像取得に失敗しました ({e})")
    print("ブラウザのカメラアクセス権限が許可されているか確認してください。")
    image_bgr = None

if image_bgr is not None:
    # face_recognition用にBGRからRGBへ変換
    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)

    # 5. AIで顔のパーツ（ランドマーク）を特定
    face_landmarks_list = face_recognition.face_landmarks(image_rgb)

    if len(face_landmarks_list) == 0:
        print("エラー: 顔が検出されませんでした。正面を向き、十分な明るさの環境で再撮影してください。")
        cv2_imshow(image_bgr)
    else:
        # 最初の1人分の顔データを処理
        landmarks = face_landmarks_list[0]

        # 左右の目の座標を取得
        left_eye = landmarks['left_eye']
        right_eye = landmarks['right_eye']

        # 両目の開き具合の平均値を算出
        left_ear = calculate_ear(left_eye)
        right_ear = calculate_ear(right_eye)
        avg_ear = (left_ear + right_ear) / 2.0

        print(f"\n================ 判定結果 ================")
        print(f"左目のEAR: {left_ear:.3f} / 右目のEAR: {right_ear:.3f}")
        print(f"平均EAR (目の開き具合): {avg_ear:.3f}")

        # 6. 眠気判定（基準値：0.22以下なら眠気・閉眼と判定）
        THRESHOLD = 0.22

        if avg_ear < THRESHOLD:
            status_text = f"Status: SLEEPY (EAR: {avg_ear:.2f})"
            color = (0, 0, 255) # 赤色 (BGR)
        else:
            status_text = f"Status: AWAKE (EAR: {avg_ear:.2f})"
            color = (0, 255, 0) # 緑色 (BGR)

        # 7. 画像に目の輪郭と判定結果を重畳描画
        # 目の輪郭をポリラインで描画
        for eye in [left_eye, right_eye]:
            pts = np.array(eye, np.int32)
            cv2.polylines(image_bgr, [pts], isClosed=True, color=(255, 200, 0), thickness=2)

        # 結果表示用バナー背景を描画（視認性向上）
        cv2.rectangle(image_bgr, (20, 20), (450, 75), (40, 40, 40), -1)
        cv2.putText(image_bgr, status_text, (30, 58), cv2.FONT_HERSHEY_SIMPLEX, 0.9, color, 2, cv2.LINE_AA)

        # 画面に結果画像を表示
        cv2_imshow(image_bgr)
